# Exercise 5D — Capture Zone Delineation

**Course:** GEOV212 Hydrogeology  
**Prerequisites:** Exercise 5B (calibrated steady-state model) must be run first to produce
`data/model_input/exercise5_model_outputs.nc`.

## Learning objectives

After completing this exercise you will be able to:

1. Load and interpret the outputs of a calibrated groundwater model.
2. Compute the Darcy velocity field and convert it to pore (seepage) velocity.
3. Select a surface-water feature (lake or river reach) and define it as a
   capture-zone target.
4. Delineate backward-advective capture zones at multiple timescales using
   backward particle tracking (MODPATH 7 via flopy, or the built-in RK4 fallback).
5. Quantify how capture-zone area and shape respond to uncertainty in effective
   porosity (*n*ₑ).
6. Export capture-zone polygons to KML for overlay in Google Earth.
7. Write a concise hydrogeological interpretation of your results.

## What is a capture zone?

A **capture zone** is the region of an aquifer that contributes water to a given
surface-water body (or well) within a specified travel time.  It is delineated by
releasing imaginary tracer particles *at* the target feature and tracking them
**backward** along streamlines against the flow direction until they have travelled
for the chosen time period.  The envelope of particle end-points defines the
capture zone for that timescale.

Capture zones are used in:
- **Source-water protection** (drinking-water intakes, lake catchments).
- **Contamination assessment** (identifying the up-gradient source area for a
  contaminated lake or river reach).
- **Groundwater management** (defining recharge areas that must be protected).

---

## Part A — Load calibrated model outputs

We reload the NetCDF file saved at the end of Exercise 5B.  Everything needed
(head field, hydraulic conductivity, grid geometry, masks) is contained in that
single file.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import xarray as xr
import cmcrameri.cm as cmc
from mpl_toolkits.axes_grid1 import make_axes_locatable
from rasterio.transform import from_origin, rowcol
from affine import Affine

import exercise_5_gw_model_utils as gwu
import exercise_5_gw_plot_utils  as gwp

# ── Load NetCDF saved by Exercise 5B ─────────────────────────────────────────
NC_PATH = 'data/model_input/exercise5_model_outputs.nc'
ds = xr.open_dataset(NC_PATH)

# ── Reconstruct grid arrays ───────────────────────────────────────────────────
active  = ds['active'].values.astype(bool)   # (nrow, ncol)
dem     = ds['dem'].values.astype(float)      # m a.s.l.
sw      = ds['sw'].values.astype(int)         # 0=dry, 1=lake, 2=river, 3=sea
sea     = ds['sea'].values.astype(bool)
geology = ds['geology'].values.astype(int)

# Select 'geology' calibration head field as default
# (change 'head_geology' → 'head_uniform' if you ran only the uniform scenario)
BEST_SCENARIO = str(ds.attrs.get('best_calibration', 'geology'))
head_key = f'head_{BEST_SCENARIO}'
if head_key not in ds:
    head_key = [k for k in ds.data_vars if k.startswith('head_')][0]
    print(f'Warning: best_calibration key not found; using {head_key}')
head = ds[head_key].values.astype(float)       # hydraulic head (m a.s.l.)
hk   = ds['best_hk'].values.astype(float)      # hydraulic conductivity (m/s)

# ── Grid geometry ─────────────────────────────────────────────────────────────
nrow, ncol          = active.shape
cell_size_m         = float(ds.attrs['cell_size_m'])
delr = delc         = cell_size_m                # square cells
aquifer_thickness_m = float(ds.attrs['aquifer_thickness_m'])

# Reconstruct the Affine transform from dataset coordinates
x_min = float(ds.coords['x'].values[0])  - cell_size_m / 2.0
y_max = float(ds.coords['y'].values[0])  + cell_size_m / 2.0
transform = Affine(cell_size_m, 0.0, x_min, 0.0, -cell_size_m, y_max)

sw_cells = (sw > 0) & (sw < 3)   # lakes + rivers, not sea
is_sea   = sea.astype(bool)

# ── Grid dict (same pattern as Exercise 5B) ───────────────────────────────────
grid = dict(
    dem=dem, active=active, sw_cells=sw_cells, is_sea=is_sea,
    nrow=nrow, ncol=ncol, delr=delr, delc=delc,
    transform=transform, aquifer_thickness_m=aquifer_thickness_m,
)

print(f'Loaded scenario : {head_key}')
print(f'Grid            : {nrow} rows × {ncol} cols  ({cell_size_m:.0f} m cells)')
print(f'Active cells    : {active.sum():,}')
print(f'SW cells (lake/river): {sw_cells.sum():,}')
print(f'Head range      : {np.nanmin(head[active]):.1f} – {np.nanmax(head[active]):.1f} m a.s.l.')


## Part B — Select a surface-water target feature

**Your task:** choose which surface-water body will be the capture-zone target.
Set `TARGET_ROW` and `TARGET_COL` to any grid cell that lies on a lake or river
(`sw == 1` or `sw == 2`).  The code will automatically expand the selection to
all connected SW cells that form the same water body.

You can find a suitable cell by inspecting the map plotted below.
Alternatively, if you know the projected coordinates (EPSG:25833), set
`TARGET_X` and `TARGET_Y` instead and set `USE_COORDS = True`.

In [ ]:
# ── Student input ─────────────────────────────────────────────────────────────

# Option 1: specify a grid cell (row, col) — default
USE_COORDS = False    # set True to use projected coordinates instead
TARGET_ROW = 50       # <-- change this
TARGET_COL = 60       # <-- change this

# Option 2: projected coordinates (EPSG:25833, metres)
TARGET_X = None   # e.g. 460000.0
TARGET_Y = None   # e.g. 6850000.0

# Maximum flood-fill distance from the seed cell (cells).
# Increase to capture larger connected water bodies.
MAX_FILL_RADIUS = 200

# ── Convert projected coords → row/col if needed ──────────────────────────────
if USE_COORDS and TARGET_X is not None and TARGET_Y is not None:
    from rasterio.transform import rowcol as _rowcol
    _r, _c = _rowcol(transform, TARGET_X, TARGET_Y)
    TARGET_ROW = int(np.clip(_r, 0, nrow - 1))
    TARGET_COL = int(np.clip(_c, 0, ncol - 1))
    print(f'Projected ({TARGET_X}, {TARGET_Y}) → row {TARGET_ROW}, col {TARGET_COL}')

# ── Validate seed cell ────────────────────────────────────────────────────────
if not sw_cells[TARGET_ROW, TARGET_COL]:
    # Snap to nearest SW cell
    sw_r, sw_c = np.where(sw_cells)
    dists = np.hypot(sw_r - TARGET_ROW, sw_c - TARGET_COL)
    nearest = np.argmin(dists)
    TARGET_ROW, TARGET_COL = int(sw_r[nearest]), int(sw_c[nearest])
    print(f'Seed snapped to nearest SW cell: row {TARGET_ROW}, col {TARGET_COL}')

# ── BFS flood-fill: collect all connected SW cells (4-connectivity) ───────────
from collections import deque

def _flood_fill_sw(sw_mask, seed_r, seed_c, max_steps=200):
    """Return set of (row, col) for all SW cells connected to the seed."""
    nrow_, ncol_ = sw_mask.shape
    visited = set()
    queue   = deque([(seed_r, seed_c, 0)])
    while queue:
        r, c, depth = queue.popleft()
        if (r, c) in visited or depth > max_steps:
            continue
        if not (0 <= r < nrow_ and 0 <= c < ncol_):
            continue
        if not sw_mask[r, c]:
            continue
        visited.add((r, c))
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            queue.append((r+dr, c+dc, depth+1))
    return visited

target_cells = _flood_fill_sw(sw_cells, TARGET_ROW, TARGET_COL, MAX_FILL_RADIUS)
target_mask  = np.zeros((nrow, ncol), dtype=bool)
for r, c in target_cells:
    target_mask[r, c] = True

print(f'Target seed     : row {TARGET_ROW}, col {TARGET_COL}  (sw={sw[TARGET_ROW, TARGET_COL]})')
print(f'Connected cells : {len(target_cells)}  '
      f'(area ≈ {len(target_cells)*cell_size_m**2/1e4:.1f} ha)')

# ── Overview map: highlight the target feature ────────────────────────────────
ph = gwp._panel_h(nrow, ncol)
fig, ax = plt.subplots(figsize=(7, ph))
head_plot = np.where(active, head, np.nan)
vlo = np.nanpercentile(head_plot[active], 2)
vhi = np.nanpercentile(head_plot[active], 98)
cf = ax.contourf(np.arange(ncol), np.arange(nrow), np.ma.masked_invalid(head_plot),
                 levels=np.linspace(vlo, vhi, 21), cmap=cmc.lapaz, extend='both')
gwp._cbar(cf, ax, 'Head (m a.s.l.)')

# SW overlay
sw_ov = np.ma.masked_where(~sw_cells, np.ones((nrow, ncol)))
ax.imshow(sw_ov, cmap='Blues', vmin=0, vmax=1, alpha=0.45, zorder=4)

# Target feature
tgt_ov = np.ma.masked_where(~target_mask, np.ones((nrow, ncol)))
ax.imshow(tgt_ov, cmap='Reds', vmin=0, vmax=1, alpha=0.85, zorder=5)
ax.plot(TARGET_COL, TARGET_ROW, 'r*', ms=14, zorder=7, label='Seed cell')

gwp.add_map_ticks(ax, grid)
gwp.add_map_overlays(ax, grid)
ax.set_title('Target surface-water feature (red) — water-table elevation background')
ax.legend(fontsize=8, framealpha=0.8)
plt.show()


## Part C — Darcy velocity field and pore velocity

The groundwater flow velocity at every cell is computed in two steps:

1. **Darcy flux** (specific discharge) $q$ [m/s]:  
   $q_x = -K \, \frac{\partial h}{\partial x}$, &nbsp; $q_y = -K \, \frac{\partial h}{\partial y}$

2. **Pore (seepage) velocity** $v$ [m/s]:  
   $v = q / n_e$

where $n_e$ is the **effective porosity** (set in Part D).  The pore velocity is
what governs advective solute transport — it is the velocity actually experienced
by a water parcel moving through the pore space.

In [ ]:
# ── Compute Darcy flux field ──────────────────────────────────────────────────
qx, qy, q_mag = gwu.compute_darcy_flux(head, hk, delr, delc, active)

_S_PER_YR = 365.25 * 86400.0
q_myr = np.where(active, q_mag * _S_PER_YR, np.nan)   # |q| in m/yr for display

# ── Plot: Darcy flux magnitude + flow arrows ──────────────────────────────────
ph = gwp._panel_h(nrow, ncol)
fig, axes = plt.subplots(1, 2, figsize=(13, ph))
fig.subplots_adjust(left=0.07, right=0.96, top=0.94, bottom=0.06, wspace=0.45)

# Left: Darcy flux magnitude
vq = float(np.nanpercentile(q_myr[active], 98))
im0 = axes[0].imshow(q_myr, cmap='YlOrBr', vmin=0, vmax=vq)
gwp._cbar(im0, axes[0], '|q| (m/yr)')
u_p = np.nan_to_num(qx, nan=0.0)
v_p = np.nan_to_num(qy, nan=0.0)
try:
    axes[0].streamplot(np.arange(ncol), np.arange(nrow), u_p, v_p,
                       color='k', linewidth=0.8, arrowsize=1.2,
                       density=0.65, zorder=5)
except Exception:
    step = max(1, nrow // 20)
    axes[0].quiver(np.arange(0, ncol, step), np.arange(0, nrow, step),
                   u_p[::step, ::step], -v_p[::step, ::step],
                   color='k', scale=None, width=0.003, zorder=5)
axes[0].set_title('A) Darcy flux magnitude (m/yr) + flow arrows')
gwp.add_map_ticks(axes[0], grid)
gwp.add_map_overlays(axes[0], grid)

# Right: log10 of hydraulic conductivity
T_log = np.where(active & (hk > 0),
                 np.log10(np.maximum(hk * aquifer_thickness_m, 1e-12)), np.nan)
im1 = axes[1].imshow(T_log, cmap=cmc.batlow)
gwp._cbar(im1, axes[1], 'log₁₀(T [m²/s])')
axes[1].set_title('B) log₁₀ Transmissivity (m²/s)')
gwp.add_map_ticks(axes[1], grid)
gwp.add_map_overlays(axes[1], grid)

fig.suptitle('Darcy velocity field', fontsize=11)
plt.show()

print(f'Median |q| over active domain : {np.nanmedian(q_myr[active]):.2f} m/yr')
print(f'98th-percentile |q|           : {vq:.2f} m/yr')


## Part D — Parameters: timescales and effective porosity

### Effective porosity (*n*ₑ)

Effective porosity is the fraction of the aquifer volume through which water
actually flows (i.e., connected, drainable pore space).  It is smaller than
total porosity because isolated pores and dead-end pores do not contribute to
advective transport.

| Material | Typical *n*ₑ range | Representative value |
|---|---|---|
| Gravel | 0.15 – 0.35 | 0.25 |
| Coarse sand | 0.20 – 0.35 | 0.28 |
| Medium sand | 0.15 – 0.30 | 0.25 |
| Fine sand | 0.10 – 0.25 | 0.20 |
| Sandy till / moraine | 0.05 – 0.20 | 0.12 |
| Silty till | 0.02 – 0.10 | 0.05 |
| Fractured crystalline bedrock | 0.001 – 0.05 | 0.01 |
| Massive crystalline bedrock | 0.0001 – 0.005 | 0.001 |

*Sources: Domenico & Schwartz (1998); Fetter (2001); Freeze & Cherry (1979).*

**Your task:**
- Choose 2–3 effective porosity values appropriate for the geology in your study
  area (inspect the geology map from Exercise 5A/5B).
- Choose 2–3 capture-time horizons (in years) that are hydrologically meaningful
  for the aquifer scale.

Edit the two lists below.

In [ ]:
# ── Student input ─────────────────────────────────────────────────────────────

# Effective porosity scenarios
# Choose 2–4 values spanning the plausible range for your geology.
ne_scenarios = [0.05, 0.15, 0.30]   # <-- edit these

# Backward tracking timescales (years)
# Particles are released at the target feature and tracked backward for each duration.
time_horizons_yr = [10, 50, 100]    # <-- edit these

# Particle seeding density: one particle per N cells of the target feature.
# Lower = finer capture zone outline (slower); higher = faster but coarser.
PARTICLES_PER_CELL = 4   # 4 particles per SW cell (2×2 sub-cell grid)

# ── Derived constants ─────────────────────────────────────────────────────────
n_ne    = len(ne_scenarios)
n_times = len(time_horizons_yr)

print('Effective porosity scenarios :', ne_scenarios)
print('Time horizons (yr)           :', time_horizons_yr)
print(f'Target cells                 : {len(target_cells)}')
n_particles = len(target_cells) * PARTICLES_PER_CELL
print(f'Total particles per scenario : {n_particles:,}')

# Quick geology distribution in target area
tgt_rows, tgt_cols = zip(*target_cells)
geo_ids, geo_counts = np.unique(geology[list(tgt_rows), list(tgt_cols)],
                                 return_counts=True)
print('\nGeology units in target feature:')
for gid, cnt in zip(geo_ids, geo_counts):
    print(f'  Unit {gid:3d} : {cnt} cells ({cnt/len(target_cells):.0%})')


## Part E — Backward particle tracking

We implement a pure-Python backward RK4 particle tracker that works directly
from the Darcy velocity arrays.  This avoids any external MODFLOW/MODPATH
dependency while remaining physically correct for the steady-state, single-layer
model.

**Method:** Particles are seeded at the target SW cells and integrated *backward*
in time (i.e., against the flow direction) using a 4th-order Runge-Kutta scheme
with the pore velocity field $v = q / n_e$.  Integration stops when:
- The requested travel time is exceeded, **or**
- The particle leaves the active domain.

The set of particle trajectories defines the capture zone for a given (*t*, *n*ₑ)
combination.

In [ ]:
from scipy.interpolate import RegularGridInterpolator

def _make_velocity_interpolators(qx_arr, qy_arr, active_arr, ne):
    """
    Build bilinear interpolators for the pore velocity components.

    qx_arr / qy_arr : Darcy flux arrays (m/s), positive = East / North.
    active_arr      : Boolean mask.
    ne              : Effective porosity (dimensionless).

    Returns (interp_vx, interp_vy) as callables that accept (row, col) pairs
    in continuous grid coordinates.
    """
    # Pore velocity (backward tracking → negate the velocity)
    vx = np.where(active_arr, qx_arr / ne, 0.0)   # East (+col) component
    vy = np.where(active_arr, qy_arr / ne, 0.0)   # North (−row) component

    rows_ax = np.arange(qx_arr.shape[0], dtype=float)
    cols_ax = np.arange(qx_arr.shape[1], dtype=float)

    interp_vx = RegularGridInterpolator(
        (rows_ax, cols_ax), vx,
        method='linear', bounds_error=False, fill_value=0.0)
    interp_vy = RegularGridInterpolator(
        (rows_ax, cols_ax), vy,
        method='linear', bounds_error=False, fill_value=0.0)
    return interp_vx, interp_vy


def _rk4_step_backward(r, c, dt_s, interp_vx, interp_vy, delr, delc):
    """
    Single RK4 backward step in grid-coordinate space.

    The velocity is in m/s; grid coordinates use cell indices so we divide
    by cell size to convert velocity to (index units)/s.

    Returns (r_new, c_new) after one backward step of dt_s seconds.
    """
    def deriv(r_, c_):
        pt = np.array([[r_, c_]])
        vx = float(interp_vx(pt))   # m/s, East  →  positive c direction
        vy = float(interp_vy(pt))   # m/s, North → negative r direction
        dr = -(-vy / delr)          # backward: negate velocity, then convert
        dc = -(vx  / delc)          # backward: negate velocity, then convert
        return dr, dc

    k1r, k1c = deriv(r,               c)
    k2r, k2c = deriv(r + 0.5*dt_s*k1r, c + 0.5*dt_s*k1c)
    k3r, k3c = deriv(r + 0.5*dt_s*k2r, c + 0.5*dt_s*k2c)
    k4r, k4c = deriv(r +     dt_s*k3r, c +     dt_s*k3c)

    r_new = r + (dt_s / 6.0) * (k1r + 2*k2r + 2*k3r + k4r)
    c_new = c + (dt_s / 6.0) * (k1c + 2*k2c + 2*k3c + k4c)
    return r_new, c_new


def track_particles_backward(qx_arr, qy_arr, active_arr, seeds_rc,
                              travel_time_s, ne,
                              delr, delc, dt_fraction=0.25):
    """
    Track a set of particles backward from seeds for *travel_time_s* seconds.

    Parameters
    ----------
    qx_arr, qy_arr  : 2-D Darcy flux arrays (m/s).
    active_arr      : Boolean active-cell mask.
    seeds_rc        : (N, 2) array of (row, col) seed positions in continuous
                      grid coords (particles start here at t=0).
    travel_time_s   : Total backward tracking time (seconds).
    ne              : Effective porosity.
    delr, delc      : Cell sizes (m).
    dt_fraction     : Step size as fraction of cell transit time (default 0.25).

    Returns
    -------
    end_rc : (N, 2) array of final (row, col) positions (NaN if left domain).
    paths  : list of (M_i, 2) arrays — one trajectory per particle.
    """
    nrow_, ncol_ = active_arr.shape
    interp_vx, interp_vy = _make_velocity_interpolators(qx_arr, qy_arr,
                                                         active_arr, ne)

    # Adaptive time step: fraction of shortest cell-transit time
    q_max = float(np.nanmax(np.abs(np.concatenate([qx_arr[active_arr],
                                                    qy_arr[active_arr]]))))
    q_max = max(q_max, 1e-12)
    v_max = q_max / ne
    dt_s  = dt_fraction * min(delr, delc) / v_max
    dt_s  = min(dt_s, travel_time_s / 10.0)   # at least 10 steps
    n_steps = max(1, int(np.ceil(travel_time_s / dt_s)))

    seeds_rc = np.asarray(seeds_rc, dtype=float)
    n_part   = seeds_rc.shape[0]
    pos      = seeds_rc.copy()          # current positions
    active_p = np.ones(n_part, dtype=bool)
    paths    = [[p.copy()] for p in pos]

    t_elapsed = 0.0
    for _ in range(n_steps):
        dt_actual = min(dt_s, travel_time_s - t_elapsed)
        if dt_actual <= 0:
            break
        for i in range(n_part):
            if not active_p[i]:
                continue
            r_new, c_new = _rk4_step_backward(
                pos[i, 0], pos[i, 1], dt_actual,
                interp_vx, interp_vy, delr, delc)
            # Boundary check
            ri, ci = int(round(r_new)), int(round(c_new))
            if (ri < 0 or ri >= nrow_ or ci < 0 or ci >= ncol_
                    or not active_arr[ri, ci]):
                active_p[i] = False
                pos[i] = np.nan
            else:
                pos[i, 0], pos[i, 1] = r_new, c_new
                paths[i].append(pos[i].copy())
        t_elapsed += dt_actual

    # Convert path lists to arrays
    path_arrays = [np.vstack(p) for p in paths]
    end_rc = np.array([p[-1] if len(p) > 0 else [np.nan, np.nan]
                       for p in path_arrays])
    return end_rc, path_arrays


def _seed_particles(target_mask, ppc, jitter_scale=0.35):
    """
    Generate seed particle positions on target SW cells.

    ppc          : particles per cell (1, 4, or 9 for regular sub-grids).
    jitter_scale : fraction of cell size for random jitter (reduces grid artefacts).
    """
    rng = np.random.default_rng(42)
    seeds = []
    sqppc = int(np.round(np.sqrt(ppc)))
    offsets = np.linspace(-0.4, 0.4, sqppc) if sqppc > 1 else [0.0]
    for r, c in zip(*np.where(target_mask)):
        for dr in offsets:
            for dc in offsets:
                jr = dr + rng.uniform(-jitter_scale/sqppc, jitter_scale/sqppc)
                jc = dc + rng.uniform(-jitter_scale/sqppc, jitter_scale/sqppc)
                seeds.append([r + jr, c + jc])
    return np.array(seeds)


seeds_rc = _seed_particles(target_mask, PARTICLES_PER_CELL)
print(f'Seeded {len(seeds_rc):,} particles on {target_mask.sum()} target cells')


In [ ]:
# ── Run all (time, ne) combinations ──────────────────────────────────────────
# Results stored in a dict: results[(t_yr, ne)] = {'ends': ..., 'paths': ...}

_S_PER_YR = 365.25 * 86400.0

results = {}
total_runs = n_times * n_ne
run_no = 0

print(f'Running {total_runs} backward tracking scenarios …\n')
print(f'  {"ne":>6}  {"time (yr)":>10}  {"particles":>10}  {"reached boundary":>18}')
print('  ' + '-' * 52)

for t_yr in time_horizons_yr:
    for ne in ne_scenarios:
        travel_s = t_yr * _S_PER_YR
        end_rc, paths = track_particles_backward(
            qx, qy, active, seeds_rc,
            travel_time_s=travel_s,
            ne=ne, delr=delr, delc=delc,
            dt_fraction=0.25,
        )
        n_alive   = np.sum(np.all(np.isfinite(end_rc), axis=1))
        n_exited  = len(end_rc) - n_alive
        results[(t_yr, ne)] = {'ends': end_rc, 'paths': paths}
        run_no += 1
        print(f'  {ne:6.3f}  {t_yr:10d}  {len(end_rc):10,}  '
              f'{n_exited:15,} ({n_exited/len(end_rc):.0%})')

print('\nDone.')


## Part F — Delineate and visualise capture zones

For each scenario we rasterise the particle end-point cloud onto the model grid
(any cell containing at least one particle end-point is considered within the
capture zone), then compute the convex-hull outline for display.  Results are
shown as a grid of maps: rows = time horizons, columns = *n*ₑ scenarios.

In [ ]:
from scipy.spatial import ConvexHull

def _rasterise_endpoints(end_rc, nrow_, ncol_, min_particles=1):
    """
    Convert particle end-points to a boolean capture-zone raster.

    A cell is inside the capture zone if it contains >= min_particles endpoints.
    """
    cz = np.zeros((nrow_, ncol_), dtype=int)
    valid = end_rc[np.all(np.isfinite(end_rc), axis=1)]
    if len(valid) == 0:
        return cz.astype(bool)
    ri = np.clip(np.round(valid[:, 0]).astype(int), 0, nrow_ - 1)
    ci = np.clip(np.round(valid[:, 1]).astype(int), 0, ncol_ - 1)
    for r, c in zip(ri, ci):
        cz[r, c] += 1
    return cz >= min_particles


# Pre-compute rasters and hull outlines
cz_rasters = {}    # (t_yr, ne) → bool raster
cz_hulls   = {}    # (t_yr, ne) → hull outline (col, row) pairs or None
cz_areas   = {}    # (t_yr, ne) → area (ha)

for t_yr in time_horizons_yr:
    for ne in ne_scenarios:
        end_rc = results[(t_yr, ne)]['ends']
        cz = _rasterise_endpoints(end_rc, nrow, ncol, min_particles=1)
        cz_rasters[(t_yr, ne)] = cz
        cz_areas[(t_yr, ne)]   = float(cz.sum()) * cell_size_m**2 / 1e4  # ha

        # Convex hull in grid coordinates for outline overlay
        valid = end_rc[np.all(np.isfinite(end_rc), axis=1)]
        if len(valid) >= 4:
            try:
                pts  = valid[:, ::-1]   # (col, row) for plotting
                hull = ConvexHull(pts)
                hull_pts = np.vstack([pts[hull.vertices], pts[hull.vertices[0]]])
                cz_hulls[(t_yr, ne)] = hull_pts
            except Exception:
                cz_hulls[(t_yr, ne)] = None
        else:
            cz_hulls[(t_yr, ne)] = None

print('Capture zone areas (ha):')
print(f'  {"ne →":10s}', ''.join(f'{ne:>10.3f}' for ne in ne_scenarios))
for t_yr in time_horizons_yr:
    row_str = '  '.join(f'{cz_areas[(t_yr, ne)]:9.1f}' for ne in ne_scenarios)
    print(f'  {t_yr} yr       {row_str}')


In [ ]:
# ── Grid of capture zone maps ─────────────────────────────────────────────────

# Colour palette for time horizons
_TIME_COLOURS  = ['#e74c3c', '#e67e22', '#27ae60', '#2980b9', '#8e44ad']
_TIME_CMAPS    = ['Reds', 'Oranges', 'Greens', 'Blues', 'Purples']

ph = gwp._panel_h(nrow, ncol)
fig, axes = plt.subplots(
    n_times, n_ne,
    figsize=(5.5 * n_ne, ph * n_times),
    squeeze=False,
)
fig.subplots_adjust(left=0.06, right=0.97, top=0.95, bottom=0.03,
                    hspace=0.45, wspace=0.35)

# Background: water-table elevation
head_bg = np.where(active, head, np.nan)
vlo = np.nanpercentile(head_bg[active], 2)
vhi = np.nanpercentile(head_bg[active], 98)
_levels = np.linspace(vlo, vhi, 21)
_x, _y  = np.arange(ncol), np.arange(nrow)

for i_t, t_yr in enumerate(time_horizons_yr):
    tcol  = _TIME_COLOURS[i_t % len(_TIME_COLOURS)]
    tcmap = _TIME_CMAPS[i_t % len(_TIME_CMAPS)]

    for j_ne, ne in enumerate(ne_scenarios):
        ax = axes[i_t][j_ne]
        cz = cz_rasters[(t_yr, ne)]

        # Background head contours
        cf = ax.contourf(_x, _y, np.ma.masked_invalid(head_bg),
                         levels=_levels, cmap=cmc.lapaz, extend='both', alpha=0.55)
        ax.contour(_x, _y, np.ma.masked_invalid(head_bg),
                   levels=_levels, colors='k', linewidths=0.15, alpha=0.3)

        # Capture zone raster
        cz_show = np.ma.masked_where(~cz, np.ones((nrow, ncol)))
        ax.imshow(cz_show, cmap=tcmap, vmin=0, vmax=1, alpha=0.65, zorder=4)

        # Convex hull outline
        if cz_hulls[(t_yr, ne)] is not None:
            hull_pts = cz_hulls[(t_yr, ne)]
            ax.plot(hull_pts[:, 0], hull_pts[:, 1],
                    color=tcol, lw=1.8, zorder=6, label='Hull')

        # Target feature
        tgt_ov = np.ma.masked_where(~target_mask, np.ones((nrow, ncol)))
        ax.imshow(tgt_ov, cmap='autumn', vmin=0, vmax=1, alpha=0.9, zorder=7)

        area_ha = cz_areas[(t_yr, ne)]
        ax.set_title(
            f't = {t_yr} yr,  nₑ = {ne}\n'
            f'Area = {area_ha:.1f} ha',
            fontsize=8, color=tcol)

        gwp.add_map_ticks(ax, grid)
        gwp.add_map_overlays(ax, grid)

fig.suptitle(
    'Capture zones — backward advective particle tracking\n'
    '(shaded area = cells reached within travel time;  outline = convex hull)',
    fontsize=10)
plt.show()


## Part G — Area summary table and sensitivity analysis

We now quantify how the capture-zone area changes across scenarios and examine
the sensitivity to both travel time and effective porosity.

In [ ]:
# ── Summary DataFrame ─────────────────────────────────────────────────────────
rows = []
for t_yr in time_horizons_yr:
    for ne in ne_scenarios:
        cz = cz_rasters[(t_yr, ne)]
        end_rc = results[(t_yr, ne)]['ends']
        valid  = end_rc[np.all(np.isfinite(end_rc), axis=1)]
        n_exit = len(end_rc) - len(valid)

        # Approximate ellipse axes from covariance of end-point cloud
        if len(valid) >= 4:
            pts_m = valid * cell_size_m   # convert to metres
            cov   = np.cov(pts_m[:, 1], pts_m[:, 0])  # (easting, northing)
            eigvals = np.linalg.eigvalsh(cov)
            eigvals = np.maximum(eigvals, 0)
            semi_a  = float(2.0 * np.sqrt(np.max(eigvals)))
            semi_b  = float(2.0 * np.sqrt(np.min(eigvals)))
        else:
            semi_a = semi_b = np.nan

        rows.append(dict(
            travel_time_yr  = t_yr,
            ne              = ne,
            area_ha         = cz_areas[(t_yr, ne)],
            area_km2        = cz_areas[(t_yr, ne)] / 100.0,
            n_cells         = int(cz.sum()),
            particles_total = len(end_rc),
            particles_exited= int(n_exit),
            pct_exited      = 100.0 * n_exit / max(len(end_rc), 1),
            semi_axis_a_m   = semi_a,
            semi_axis_b_m   = semi_b,
        ))

df_summary = pd.DataFrame(rows)

# Pivot for display
pivot_area = df_summary.pivot(index='travel_time_yr', columns='ne', values='area_ha')
pivot_area.index.name = 'Travel time (yr)'
pivot_area.columns = [f'nₑ = {v}' for v in pivot_area.columns]
print('Capture zone area (ha):')
display(pivot_area.style.format('{:.1f}').background_gradient(cmap='YlOrRd', axis=None))

pivot_km2 = df_summary.pivot(index='travel_time_yr', columns='ne', values='area_km2')
pivot_km2.index.name = 'Travel time (yr)'
pivot_km2.columns = [f'nₑ = {v}' for v in pivot_km2.columns]
print('\nCapture zone area (km²):')
display(pivot_km2.style.format('{:.3f}').background_gradient(cmap='YlGnBu', axis=None))


In [ ]:
# ── Sensitivity charts ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.subplots_adjust(left=0.09, right=0.97, top=0.90, bottom=0.12, wspace=0.40)

# Left: area vs. travel time for each ne
_NE_COLOURS = ['#2980b9', '#27ae60', '#e74c3c', '#e67e22', '#8e44ad']
for j, ne in enumerate(ne_scenarios):
    sub = df_summary[df_summary['ne'] == ne].sort_values('travel_time_yr')
    axes[0].plot(sub['travel_time_yr'], sub['area_ha'],
                 'o-', color=_NE_COLOURS[j % len(_NE_COLOURS)],
                 lw=2, ms=7, label=f'nₑ = {ne}')
axes[0].set_xlabel('Travel time (yr)')
axes[0].set_ylabel('Capture zone area (ha)')
axes[0].set_title('A) Area vs. travel time')
axes[0].legend(fontsize=8, framealpha=0.85)
axes[0].grid(True, alpha=0.3)

# Right: bar chart — area for fixed travel time, varying ne
# Use the median (central) time horizon
mid_t = time_horizons_yr[len(time_horizons_yr) // 2]
sub_t = df_summary[df_summary['travel_time_yr'] == mid_t]
x_pos = np.arange(len(ne_scenarios))
bar_colours = [_NE_COLOURS[j % len(_NE_COLOURS)] for j in range(len(ne_scenarios))]
bars = axes[1].bar(x_pos, sub_t['area_ha'].values,
                   color=bar_colours, edgecolor='white', linewidth=0.8, width=0.6)
for bar, val in zip(bars, sub_t['area_ha'].values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.0f} ha', ha='center', va='bottom', fontsize=8)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels([f'nₑ = {ne}' for ne in ne_scenarios])
axes[1].set_ylabel('Capture zone area (ha)')
axes[1].set_title(f'B) Area vs. nₑ  (t = {mid_t} yr)')
axes[1].grid(axis='y', alpha=0.3)

fig.suptitle('Capture zone sensitivity analysis', fontsize=11)
plt.show()

# Full detail table
print('\nFull summary table:')
display(df_summary[['travel_time_yr', 'ne', 'area_ha', 'area_km2',
                     'particles_total', 'pct_exited',
                     'semi_axis_a_m', 'semi_axis_b_m']]
        .rename(columns={'travel_time_yr': 'time (yr)', 'ne': 'nₑ',
                         'area_ha': 'Area (ha)', 'area_km2': 'Area (km²)',
                         'particles_total': 'N particles',
                         'pct_exited': '% exited domain',
                         'semi_axis_a_m': 'Semi-axis a (m)',
                         'semi_axis_b_m': 'Semi-axis b (m)'})
        .style.format({
            'Area (ha)': '{:.1f}',
            'Area (km²)': '{:.3f}',
            '% exited domain': '{:.1f}',
            'Semi-axis a (m)': '{:.0f}',
            'Semi-axis b (m)': '{:.0f}',
        })
       )


## Part H — KML export for Google Earth

We reproject the capture-zone convex hull outlines from EPSG:25833 (Norwegian
UTM) to WGS 84 geographic coordinates (EPSG:4326) and write a KML file that can
be opened directly in Google Earth.

Each time horizon gets a separate folder; *n*ₑ scenarios are represented as
separate polygons within that folder, colour-coded by time horizon.

In [ ]:
import xml.etree.ElementTree as ET
from pyproj import Transformer
import os

KML_PATH = 'data/model_input/exercise5_capture_zones.kml'

# EPSG:25833 → WGS84 (lon/lat) transformer
_transformer = Transformer.from_crs('EPSG:25833', 'EPSG:4326', always_xy=True)

def _grid_rc_to_lonlat(hull_pts_rc, transform_, transformer_):
    """
    Convert grid (row, col) outline points to (lon, lat) WGS84.

    hull_pts_rc : (N, 2) array of (col, row) in pixel coordinates.
    transform_  : Affine transform (model CRS).
    transformer_: pyproj.Transformer from model CRS to WGS84.
    """
    # hull_pts_rc columns are (col, row) for matplotlib — note the order
    cols_ = hull_pts_rc[:, 0]
    rows_ = hull_pts_rc[:, 1]
    # Cell-centre projected coords
    x_m = transform_.c + (cols_ + 0.5) * transform_.a
    y_m = transform_.f + (rows_ + 0.5) * transform_.e
    lon, lat = transformer_.transform(x_m, y_m)
    return list(zip(lon, lat))


# KML colour format: aabbggrr (Google Earth)
_KML_COLOURS = [
    'ff3333e7',   # red   (time 0)
    'ff22a0e6',   # orange
    'ff60ae27',   # green
    'ffd98029',   # blue
    'ffad44ae',   # purple
]

# ── Build KML tree ────────────────────────────────────────────────────────────
kml_root = ET.Element('kml', xmlns='http://www.opengis.net/kml/2.2')
doc = ET.SubElement(kml_root, 'Document')
ET.SubElement(doc, 'name').text = 'Exercise 5D — Capture Zones'
ET.SubElement(doc, 'description').text = (
    'Backward-advective capture zones from GEOV212 Exercise 5D. '
    'Each folder = one time horizon. Each polygon = one nₑ scenario.'
)

# Styles
for i_t, (t_yr, kml_col) in enumerate(
        zip(time_horizons_yr, _KML_COLOURS)):
    style = ET.SubElement(doc, 'Style', id=f'style_t{t_yr}')
    poly_style = ET.SubElement(style, 'PolyStyle')
    ET.SubElement(poly_style, 'color').text = kml_col
    ET.SubElement(poly_style, 'fill').text  = '1'
    ET.SubElement(poly_style, 'outline').text = '1'
    line_style = ET.SubElement(style, 'LineStyle')
    ET.SubElement(line_style, 'color').text = 'ff000000'
    ET.SubElement(line_style, 'width').text = '1.5'

# Folders
for i_t, t_yr in enumerate(time_horizons_yr):
    folder = ET.SubElement(doc, 'Folder')
    ET.SubElement(folder, 'name').text = f'{t_yr}-year capture zone'

    for j_ne, ne in enumerate(ne_scenarios):
        hull_pts = cz_hulls.get((t_yr, ne))
        if hull_pts is None:
            print(f'  Skipping ({t_yr} yr, nₑ={ne}): no hull available')
            continue

        lonlat = _grid_rc_to_lonlat(hull_pts, transform, _transformer)

        placemark = ET.SubElement(folder, 'Placemark')
        ET.SubElement(placemark, 'name').text = (
            f'{t_yr} yr  |  nₑ = {ne}  |  {cz_areas[(t_yr,ne)]:.1f} ha'
        )
        ET.SubElement(placemark, 'styleUrl').text = f'#style_t{t_yr}'
        polygon = ET.SubElement(placemark, 'Polygon')
        ET.SubElement(polygon, 'extrude').text       = '0'
        ET.SubElement(polygon, 'tessellate').text    = '1'
        ET.SubElement(polygon, 'altitudeMode').text  = 'clampToGround'
        outer = ET.SubElement(polygon, 'outerBoundaryIs')
        ring  = ET.SubElement(outer, 'LinearRing')
        coords_text = ' '.join(
            f'{lon:.6f},{lat:.6f},0' for lon, lat in lonlat
        )
        ET.SubElement(ring, 'coordinates').text = coords_text

# Write file
tree = ET.ElementTree(kml_root)
ET.indent(tree, space='  ')   # Python 3.9+
os.makedirs(os.path.dirname(KML_PATH), exist_ok=True)
tree.write(KML_PATH, encoding='utf-8', xml_declaration=True)
print(f'KML written to: {KML_PATH}')
print('Open this file in Google Earth (File → Open) to view capture zones.')


## Part I — Reflection questions

Use the results from Parts F and G to answer the following questions in your
report (~0.5–1 page).  The questions are designed to guide your interpretation;
you do not need to answer them as separate numbered items — integrate them into a
coherent hydrogeological discussion.

### Questions to address

1. **Shape and orientation of the capture zone.**  
   Does the capture zone extend uniformly in all directions, or is it elongated
   in a preferential direction?  What controls this shape — topography, geology,
   or the position of the target feature relative to recharge and discharge areas?

2. **Effect of travel time.**  
   How does the capture zone area scale with the chosen time horizons?  Is the
   growth roughly linear, sub-linear, or super-linear?  What does this imply
   about the spatial heterogeneity of groundwater velocities in your catchment?

3. **Sensitivity to effective porosity.**  
   Compare the capture zone area for your minimum and maximum *n*ₑ values.  By
   what factor does the area change?  Why does a higher *n*ₑ produce a smaller
   capture zone for the same travel time?

4. **Geological controls.**  
   Referring to the geology map from Exercise 5A/5B, explain why the capture zone
   has the shape and extent you observe.  Are there any geological units that
   appear to act as barriers or preferential flow paths?

5. **Practical implications.**  
   If this were a drinking-water source, what land-use activities within the
   capture zone might pose a contamination risk?  Which timescale and *n*ₑ
   scenario would you use for a conservative (protective) source-water protection
   zone, and why?

6. **Limitations.**  
   Identify at least two significant simplifications of this analysis compared to
   a real-world capture zone delineation (e.g., 3-D flow, transient conditions,
   dispersion, unsaturated zone, uncertain recharge).

---

### Space for your written answer

*Write your answer in the Markdown cell below.  Aim for 0.5–1 page of text.*

*(Your answer here)*